# 🧬 Lesson 14b — Standardized Docking & ProLIF Analysis
## A Solved Analysis Notebook for the Current Workflow

> **Environment:** Python 3.11 | Conda | WSL/Ubuntu | VS Code  
> **Role:** Educational analysis of the current standardized docking results

---

> ### ⚠️ Source of truth
> This notebook does **not** implement a second docking or pose-preparation pipeline.
>
> The current authoritative chain is:
>
> `configs/docking_config.yml` → `scripts/run_standardized_docking.py` → `docking/results/standardized_affinities.csv`
>
> Current Top 5 pose preparation is handled by:
>
> `scripts/prepare_top5.py`
>
> Current ProLIF analysis is handled by:
>
> `scripts/prolif_analysis.py`

The notebook reads those validated outputs for teaching and interpretation.

## What this notebook teaches

1. How the current standardized docking ranking is selected.
2. How the current Top 5 relates to the prepared docking poses.
3. How to inspect the current ProLIF interaction summary.
4. How to compare interaction counts and interaction types across the Top 5.
5. How to interpret docking scores and ProLIF results without treating them as experimental measurements.

### Current screening state

```text
117 candidate ligands
        ↓
111 validated ligands + 6 documented exceptions
        ↓
standardized Smina docking
        ↓
current Top 5
        ↓
Top 5 pose preparation
        ↓
ProLIF interaction analysis
```

## 1️⃣ Project setup

In [ ]:
import os
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "configs" / "docking_config.yml").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not locate the project root containing configs/docking_config.yml."
    )

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")

## 2️⃣ Load the authoritative configuration and ranking

In [ ]:
from src.docking_config import load_docking_config
from src.docking import get_top_hits

config = load_docking_config(PROJECT_ROOT / "configs" / "docking_config.yml")
ranking = pd.read_csv(config.report)
top5 = get_top_hits(str(config.report), n=5)

print("Target:", config.target_pdb)
print("Reference ligand:", config.reference_ligand)
print("Docking engine:", config.smina_executable)
print("Expected validated ligands:", config.expected_ligands)
print(f"Reported ligands: {len(ranking)}")

assert len(ranking) == config.expected_ligands
assert set(ranking["status"]).issubset({"ok", "existing"})
assert ranking["ligand_id"].is_unique

print("\nCurrent Top 5:")
display(top5[["ligand_id", "affinity", "status"]].reset_index(drop=True))

## 3️⃣ Verify the current Top 5 pose inputs

The standardized docking outputs live under:

`docking/results/standardized/`

The notebook checks that the five current top-ranked poses exist. It does **not** regenerate them.

In [ ]:
pose_dir = config.output_dir
top5_ids = top5["ligand_id"].astype(str).tolist()

pose_paths = {
    lid: pose_dir / f"{lid}_out.pdbqt"
    for lid in top5_ids
}

for lid, path in pose_paths.items():
    print(f"{lid:5s} -> {path} | exists={path.is_file()}")

missing_poses = [lid for lid, path in pose_paths.items() if not path.is_file()]
assert not missing_poses, f"Missing standardized Top 5 poses: {missing_poses}"

## 4️⃣ Reproduce the pose-preparation step explicitly

When ProLIF needs RDKit-readable ligand structures, the current workflow prepares the Top 5 with:

```bash
python -m scripts.prepare_top5
```

That script converts the standardized PDBQT poses, validates them against the original ligand structures, and reconstructs a pose from the ligand template when direct conversion does not preserve the required chemistry.

The resulting ProLIF-ready files are written under:

`docking/results/fixed/`

In [ ]:
fixed_dir = PROJECT_ROOT / "docking" / "results" / "fixed"

fixed_paths = {
    lid: fixed_dir / f"{lid}_fixed.sd"
    for lid in top5_ids
}

for lid, path in fixed_paths.items():
    print(f"{lid:5s} -> {path} | exists={path.is_file()}")

missing_fixed = [lid for lid, path in fixed_paths.items() if not path.is_file()]
assert not missing_fixed, f"Missing ProLIF-ready Top 5 poses: {missing_fixed}"

## 5️⃣ Load the current ProLIF results

The authoritative current ProLIF summary is:

`docking/results/prolif_summary.csv`

It is generated from the current standardized Top 5, not from the old `affinities_full.csv` results.

In [ ]:
prolif_path = PROJECT_ROOT / "docking" / "results" / "prolif_summary.csv"
prolif = pd.read_csv(prolif_path)

required_columns = {
    "ligand_id",
    "affinity",
    "n_interactions",
    "n_residues",
    "interaction_types",
}
missing_columns = required_columns - set(prolif.columns)

assert not missing_columns, f"Missing ProLIF columns: {sorted(missing_columns)}"

prolif_ids = set(prolif["ligand_id"].astype(str))
assert set(top5_ids) == prolif_ids, (
    f"ProLIF IDs {sorted(prolif_ids)} do not match current Top 5 {sorted(top5_ids)}"
)

print(f"Current ProLIF ligands: {len(prolif)}")
display(
    prolif.sort_values("affinity")
    [["ligand_id", "affinity", "n_interactions", "n_residues", "interaction_types"]]
    .reset_index(drop=True)
)

## 6️⃣ Compare interaction counts

A larger number of ProLIF interactions does **not** automatically mean a better binder. Interaction counts are structural descriptors that should be interpreted together with docking rank, interaction type, geometry, and chemical context.

In [ ]:
plot_df = prolif.sort_values("affinity").reset_index(drop=True)

plt.figure(figsize=(8, 4.5))
plt.bar(plot_df["ligand_id"], plot_df["n_interactions"])
plt.xlabel("Ligand")
plt.ylabel("Number of detected interactions")
plt.title("ProLIF interaction count — current Top 5")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 7️⃣ Inspect interaction types

The current summary stores the interaction-type counts as a compact text field. This cell expands them for easier comparison.

In [ ]:
from collections import Counter

interaction_rows = []

for _, row in prolif.sort_values("affinity").iterrows():
    parts = [part.strip() for part in str(row["interaction_types"]).split(";") if part.strip()]
    counts = {}
    for part in parts:
        name, count = part.rsplit(":", 1)
        counts[name.strip()] = int(count)
    counts["ligand_id"] = row["ligand_id"]
    interaction_rows.append(counts)

interaction_table = pd.DataFrame(interaction_rows).fillna(0)

cols = ["ligand_id"] + sorted(
    column for column in interaction_table.columns if column != "ligand_id"
)
interaction_table = interaction_table[cols]

display(interaction_table)

## 8️⃣ Reference ligand versus the current top hit

`13U` is the crystallographic reference ligand. `R11` is the current standardized docking leader.

The comparison below is descriptive only. A higher interaction count for one compound does not, by itself, establish superior biochemical potency.

In [ ]:
summary = prolif.set_index("ligand_id")

comparison = pd.DataFrame(
    {
        "Metric": [
            "Docking score (kcal/mol)",
            "ProLIF interactions",
            "Protein residues involved",
        ],
        "R11": [
            summary.loc["R11", "affinity"],
            summary.loc["R11", "n_interactions"],
            summary.loc["R11", "n_residues"],
        ],
        "13U": [
            summary.loc["13U", "affinity"],
            summary.loc["13U", "n_interactions"],
            summary.loc["13U", "n_residues"],
        ],
    }
)

display(comparison)

## 9️⃣ Scientific interpretation

The current standardized docking ranking identifies:

```text
1. R11
2. 13U
3. BAH
4. 12U
5. 607
```

The current ProLIF analysis then characterizes how each Top-5 pose interacts with the prepared Trypsin structure.

### Important interpretation rules

- Docking scores are **model-derived ranking scores**, not experimental binding affinities.
- ProLIF interaction counts describe detected structural contacts and interaction types for the analyzed poses.
- Rule-based Lipinski, Veber, and PAINS filters provide chemical-property context; they are not full ADME/Toxicity predictions.
- The historical Notebook 16 MD analysis concerns the **13U–Trypsin** reference system and does not validate the current `R11` ranking.

## ✅ Lesson 14b summary

This solved notebook is intentionally **analysis-only**.

The current reproducible workflow is:

```text
configs/docking_config.yml
        ↓
scripts/run_standardized_docking.py
        ↓
docking/results/standardized_affinities.csv
        ↓
scripts/prepare_top5.py
        ↓
docking/results/fixed/
        ↓
scripts/prolif_analysis.py
        ↓
docking/results/prolif_summary.csv
        ↓
interpretation in this notebook
```

There is no legacy `affinities_full.csv` dependency and no competing docking implementation in this notebook.